In [1]:
from os.path import basename, abspath
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg
from pyspark.sql.window import Window

In [2]:
problem_name = basename(abspath("."))

In [3]:
spark = SparkSession.builder.appName(problem_name).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/06 01:44:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
employees = spark.read\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .format("csv")\
    .load("files/employees.txt")

In [5]:
departments = spark.read\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .format("csv")\
    .load("files/departments.txt")

In [6]:
employees.createOrReplaceTempView("employees")

In [7]:
departments.createOrReplaceTempView("departments")

In [8]:
employees.show()

+------+-------------+---+
|emp_id|department_id|age|
+------+-------------+---+
|     1|          101| 30|
|     2|          102| 40|
|     3|          101| 25|
|     4|          103| 35|
|     5|          102| 50|
|     6|          101| 55|
|     7|          101| 45|
|     8|          102| 30|
|     9|          103| 60|
|    10|          101| 28|
+------+-------------+---+



In [9]:
departments.show()

+-------------+---------------+
|department_id|department_name|
+-------------+---------------+
|          101|             HR|
|          102|             IT|
|          103|        Finance|
+-------------+---------------+



In [10]:
result = employees.withColumn("average_age", 
                     avg(col("age")).over(Window.partitionBy("department_id"))
                    )\
    .filter(col("age")>col("average_age"))\
    .join(departments.alias("d"), on = "department_id", how = "inner")\
    .select("emp_id", "d.department_id", "age", "d.department_name")

In [11]:
result.show()

+------+-------------+---+---------------+
|emp_id|department_id|age|department_name|
+------+-------------+---+---------------+
|     6|          101| 55|             HR|
|     7|          101| 45|             HR|
|     5|          102| 50|             IT|
|     9|          103| 60|        Finance|
+------+-------------+---+---------------+



In [12]:
spark.sql(
    """
    with cte as (
    select e.emp_id, d.department_id, e.age, avg(age) over(partition by d.department_id) as average_age,
    d.department_name
    from employees e join departments d on e.department_id = d.department_id
    )

    select emp_id, department_id, age, department_name
    from cte 
    where age>average_age
    """
).show()

+------+-------------+---+---------------+
|emp_id|department_id|age|department_name|
+------+-------------+---+---------------+
|     6|          101| 55|             HR|
|     7|          101| 45|             HR|
|     5|          102| 50|             IT|
|     9|          103| 60|        Finance|
+------+-------------+---+---------------+



In [13]:
spark.stop()